In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from skimage.measure import find_contours

from unc_handling import UG_prompter
from DataLoader import DataLoader
from segmentation import Segmentation
from segmentation_util import combine_prompt_sets
from evaluation import Evaluator, compare_to_recontours
from scipy.ndimage import distance_transform_edt

root = r"C:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\data\LUNDPROBE\ExtendedSamples\development"
data = DataLoader(parentfolder=root,subject_nr=0,volume_of_interest="CTVT",verbose=True)
unc_handler = UG_prompter(data=data)
seg_handler = Segmentation(data=data)

recontours = data.load_recontours()
staple = data.load_consensus()

Loaded subject newAcq_050f229dc2bdb64c with volume of interest 'CTVT'
Image shape: (88, 1024, 1024), Mask shape: (88, 1024, 1024), Uncertainty map shape: (88, 1024, 1024), Ground truth shape: (88, 1024, 1024)
Image spacing (z, y, x): (2.5, 0.46880000829696655, 0.46880000829696655)
Initilialized UG_prompter for subject 0 with volume of interest 'CTVT'
Image shape: (88, 1024, 1024), Mask shape: (88, 1024, 1024), Uncertainty map shape: (88, 1024, 1024).
Initialized Segmentation for subject 0 with volume of interest 'CTVT'
Mask shape: (88, 1024, 1024)
Building SAM predictor from checkpoint: checkpoints/MedSAM2_latest.pt


c:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\sam2\modeling\sam\transformer.py:23: UserWarning: Flash Attention is disabled as it requires a GPU with Ampere (8.0) CUDA capability.
  OLD_GPU, USE_FLASH_ATTN, MATH_KERNEL_ON = get_sdpa_settings()


In [2]:
def tween_contours_3d(contours: list, weights: list, spacing, pad_voxels=(3, 50, 50)):
    
    contours = [np.asarray(c).astype(bool) for c in contours]
    shape = contours[0].shape
    weights = np.asarray(weights, dtype=np.float32)

    #Combine all contours to crop total image volume
    combined = np.zeros(shape, dtype=bool)
    for contour in contours:
        combined |= contour

    zs, ys, xs = np.where(combined)
    
    pad_z, pad_y, pad_x = pad_voxels

    z0 = max(0, zs.min() - pad_z)
    z1 = min(shape[0], zs.max() + pad_z + 1)

    y0 = max(0, ys.min() - pad_y)
    y1 = min(shape[1], ys.max() + pad_y + 1)

    x0 = max(0, xs.min() - pad_x)
    x1 = min(shape[2], xs.max() + pad_x + 1)

    #Select cropped region of interest
    crop = (slice(z0, z1), slice(y0, y1), slice(x0, x1))

    #Create empty array to hold all weighted SDFs
    new_contour_sdf = np.zeros((z1 - z0, y1 - y0, x1 - x0),dtype=np.float32)

    for contour, weight in zip(contours, weights):
        contour_crop = contour[crop]

        outside = distance_transform_edt(~contour_crop, sampling=spacing)
        inside = distance_transform_edt(contour_crop, sampling=spacing)

        sdf = outside - inside
        new_contour_sdf += sdf * weight

    # Threshold cropped fused SDF
    new_contour_crop = new_contour_sdf <= 0

    # Paste cropped result back into original volume shape
    new_contour = np.zeros(shape, dtype=bool)
    new_contour[crop] = new_contour_crop

    return new_contour

def tween_contours_2d(contours: list, weights: list, spacing, pad_voxels=(50, 50)):
    
    assert contours[0].ndim == 2, "Contours must be 2D arrays. Use tween_contours_3d for 3D contours."

    contours = [np.asarray(c).astype(bool) for c in contours]
    weights = np.asarray(weights, dtype=np.float32)

    valid_idx = [i for i, c in enumerate(contours) if np.any(c)] #Exclude observers that did not contour this slice

    if len(valid_idx) == 0:
        return np.zeros(contours[0].shape, dtype=bool)

    #Keep only observers that contain a contour
    contours = [contours[i] for i in valid_idx]
    weights = weights[valid_idx]

    #Normalize weights after possible removal of contours
    weights = weights / weights.sum()

    shape = contours[0].shape

    #Combine all contours to determine crop region
    combined = np.zeros(shape, dtype=bool)
    for contour in contours:
        combined |= contour

    ys, xs = np.where(combined)

    pad_y, pad_x = pad_voxels

    y0 = max(0, ys.min() - pad_y)
    y1 = min(shape[0], ys.max() + pad_y + 1)

    x0 = max(0, xs.min() - pad_x)
    x1 = min(shape[1], xs.max() + pad_x + 1)

    crop = (slice(y0, y1), slice(x0, x1))

    new_contour_sdf = np.zeros((y1 - y0, x1 - x0), dtype=np.float32)

    for contour, weight in zip(contours, weights):

        contour_crop = contour[crop]

        outside = distance_transform_edt(~contour_crop, sampling=spacing)
        inside = distance_transform_edt(contour_crop, sampling=spacing)

        sdf = outside - inside
        new_contour_sdf += sdf * weight

    new_contour_crop = new_contour_sdf <= 0

    new_contour = np.zeros(shape, dtype=bool)
    new_contour[crop] = new_contour_crop

    return new_contour

def stack_2d_tweening(contours, weights, spacing, pad_voxels=(50, 50)):
    
    slices = []

    for z in range(contours[0].shape[0]):
        slice_contours = [contour[z] for contour in contours]
        tweened_slice = tween_contours_2d(slice_contours, weights, spacing[1:], pad_voxels)
        slices.append(tweened_slice)
    
    return np.stack(slices, axis=0)

In [3]:
tweened_contour = tween_contours_3d(contours=data.observer_recontours, weights=[0.25, 0.25, 0.25, 0.25], spacing=data.img_spacing)

In [4]:
gt_plus_recontour0 = [data.observer_recontours[0], data.gt]

In [5]:
tweens = {
    "3d Tweening": tween_contours_3d(
        gt_plus_recontour0,
        weights=[0.5,0.5],
        spacing=data.img_spacing,
    ),
    "2d stacked Tweening": stack_2d_tweening(
        gt_plus_recontour0,
        weights=[0.5, 0.5],
        spacing=data.img_spacing,
    ),
}

In [6]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from skimage.measure import find_contours


def interactive_tweened_contour_plotter(
    data,
    tweened,
    recontours=None,
    staple=None,
    figsize=(8, 8),
    zoom_fraction=0.30,
):
    img = np.asarray(data.img)

    if isinstance(tweened, dict):
        tweened_dict = {
            name: np.asarray(mask).astype(bool)
            for name, mask in tweened.items()
        }
    else:
        tweened_dict = {
            "Tweened SDF": np.asarray(tweened).astype(bool)
        }

    if recontours is None:
        recontours = data.load_recontours()
    recontours = [np.asarray(r).astype(bool) for r in recontours]

    observer_names = getattr(
        data,
        "observer_names",
        [f"{i + 1}" for i in range(len(recontours))]
    )

    if staple is None:
        try:
            staple = np.asarray(data.load_consensus()).astype(bool)
        except Exception:
            staple = None
    else:
        staple = np.asarray(staple).astype(bool)

    dense = np.asarray(data.mask).astype(bool) if hasattr(data, "mask") else None
    gt = np.asarray(data.gt).astype(bool) if hasattr(data, "gt") else None

    n_slices = img.shape[0]

    observer_colors = [
        "red", "orange", "purple", "brown",
        "pink", "cyan", "magenta", "gray"
    ]

    tween_colors = [
        "yellow", "white", "lime", "cyan",
        "magenta", "gold", "deepskyblue", "hotpink"
    ]

    def draw_contour(ax, mask_slice, color, linewidth=1.0, linestyle="-", label=None):
        if mask_slice is None or not np.any(mask_slice):
            return

        first = True
        for contour in find_contours(mask_slice.astype(float), 0.5):
            ax.plot(
                contour[:, 1],
                contour[:, 0],
                color=color,
                linewidth=linewidth,
                linestyle=linestyle,
                label=label if first else None,
            )
            first = False

    def get_zoom_bounds(z):
        combined = np.zeros_like(img[z], dtype=bool)

        for obs in recontours:
            combined |= obs[z]

        for mask in tweened_dict.values():
            combined |= mask[z]

        if staple is not None:
            combined |= staple[z]

        if dense is not None:
            combined |= dense[z]

        if gt is not None:
            combined |= gt[z]

        if not combined.any():
            return None

        ys, xs = np.where(combined)

        y_center = int((ys.min() + ys.max()) / 2)
        x_center = int((xs.min() + xs.max()) / 2)

        H, W = img[z].shape
        half_h = int(H * zoom_fraction / 2)
        half_w = int(W * zoom_fraction / 2)

        y0 = max(0, y_center - half_h)
        y1 = min(H, y_center + half_h)
        x0 = max(0, x_center - half_w)
        x1 = min(W, x_center + half_w)

        return x0, x1, y0, y1

    slice_slider = widgets.IntSlider(
        value=n_slices // 2,
        min=0,
        max=n_slices - 1,
        step=1,
        description="Slice",
    )

    show_staple = widgets.Checkbox(value=True, description="STAPLE")
    show_observers = widgets.Checkbox(value=True, description="Observers")
    show_gt = widgets.Checkbox(value=False, description="GT")
    show_dense = widgets.Checkbox(value=False, description="nnUNet")
    zoom = widgets.Checkbox(value=True, description="Zoom")

    tween_checkboxes = {
        name: widgets.Checkbox(value=True, description=name)
        for name in tweened_dict.keys()
    }

    def update(z, show_staple, show_observers, show_gt, show_dense, zoom, **show_tweens):
        fig, ax = plt.subplots(figsize=figsize)
        ax.imshow(img[z], cmap="gray")

        if show_dense and dense is not None:
            draw_contour(ax, dense[z], "deepskyblue", 1.0, ":", "nnUNet")

        if show_gt and gt is not None:
            draw_contour(ax, gt[z], "lime", 1.2, "-", "GT")

        if show_observers:
            for i, obs in enumerate(recontours):
                draw_contour(
                    ax,
                    obs[z],
                    observer_colors[i % len(observer_colors)],
                    0.9,
                    "-",
                    f"Obs {observer_names[i]}",
                )

        if show_staple and staple is not None:
            draw_contour(ax, staple[z], "blue", 1.1, "-", "STAPLE")

        for i, (name, mask) in enumerate(tweened_dict.items()):
            if show_tweens.get(name, False):
                draw_contour(
                    ax,
                    mask[z],
                    tween_colors[i % len(tween_colors)],
                    1.3,
                    "-",
                    name,
                )

        if zoom:
            bounds = get_zoom_bounds(z)
            if bounds is not None:
                x0, x1, y0, y1 = bounds
                ax.set_xlim(x0, x1)
                ax.set_ylim(y1, y0)

        subject_name = getattr(data, "subject_name", "Subject")
        ax.set_title(f"{subject_name} | slice {z}")
        ax.axis("off")

        handles, labels = ax.get_legend_handles_labels()
        unique = dict(zip(labels, handles))
        if unique:
            ax.legend(unique.values(), unique.keys(), loc="upper right")

        plt.show()

    controls = {
        "z": slice_slider,
        "show_staple": show_staple,
        "show_observers": show_observers,
        "show_gt": show_gt,
        "show_dense": show_dense,
        "zoom": zoom,
    }

    controls.update(tween_checkboxes)

    ui = widgets.VBox([
        slice_slider,
        widgets.HBox([show_staple, show_observers]),
        widgets.HBox([show_gt, show_dense, zoom]),
        widgets.VBox(list(tween_checkboxes.values())),
    ])

    out = widgets.interactive_output(update, controls)

    display(ui, out)

In [ ]:
interactive_tweened_contour_plotter(
    data=data,
    tweened=tweens,
    recontours=[data.observer_recontours[0]],
)

Output()

In [25]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from scipy.ndimage import distance_transform_edt


def mask_to_sdf(mask, spacing=None):
    mask = np.asarray(mask).astype(bool)

    outside = distance_transform_edt(~mask, sampling=spacing)
    inside = distance_transform_edt(mask, sampling=spacing)

    return outside - inside  # negative inside, positive outside


def interactive_sdf_tween_plotter(
    seg_a,
    seg_b,
    spacing=None,
    name_a="Segmentation A",
    name_b="Segmentation B",
    t_init=0.5,
    zoom_fraction=0.15,
    figsize=(15, 5),
):

    seg_a = np.asarray(seg_a).astype(bool)
    seg_b = np.asarray(seg_b).astype(bool)

    if seg_a.shape != seg_b.shape:
        raise ValueError(
            f"seg_a and seg_b must have same shape. "
            f"Got {seg_a.shape} and {seg_b.shape}"
        )

    if seg_a.ndim != 3:
        raise ValueError(
            f"Expected 3D segmentations with shape (z,y,x). "
            f"Got {seg_a.shape}"
        )

    n_slices = seg_a.shape[0]

    spacing_2d = spacing[1:] if spacing is not None else None

    def get_zoom_bounds(mask_a, mask_b):

        combined = mask_a | mask_b

        if not np.any(combined):
            return None

        ys, xs = np.where(combined)

        y0 = ys.min()
        y1 = ys.max()

        x0 = xs.min()
        x1 = xs.max()

        h, w = combined.shape

        margin_y = int((y1 - y0 + 1) * zoom_fraction)
        margin_x = int((x1 - x0 + 1) * zoom_fraction)

        y0 = max(0, y0 - margin_y)
        y1 = min(h, y1 + margin_y)

        x0 = max(0, x0 - margin_x)
        x1 = min(w, x1 + margin_x)

        return x0, x1, y0, y1

    def update(z, t, clip_percentile, show_zero_contour, zoom):

        mask_a = seg_a[z]
        mask_b = seg_b[z]

        sdf_a = mask_to_sdf(mask_a, spacing=spacing_2d)
        sdf_b = mask_to_sdf(mask_b, spacing=spacing_2d)

        sdf_t = (1 - t) * sdf_a + t * sdf_b

        vmin = -45
        vmax = 45

        fig, axes = plt.subplots(1, 3, figsize=figsize)

        plots = [
            (sdf_a, name_a),
            (sdf_b, name_b),
            (sdf_t, f"Tweened SDF (t={t:.2f})"),
        ]

        for ax, (sdf, title) in zip(axes, plots):

            im = ax.imshow(
                sdf,
                cmap="seismic",
                vmin=-vmax,
                vmax=vmax,
                interpolation="none",
            )

            if show_zero_contour:
                ax.contour(
                    sdf,
                    levels=[0],
                    colors="black",
                    linewidths=1.0,
                )

            if zoom:
                bounds = get_zoom_bounds(mask_a, mask_b)

                if bounds is not None:
                    x0, x1, y0, y1 = bounds

                    ax.set_xlim(x0, x1)
                    ax.set_ylim(y1, y0)

            ax.set_title(title)
            ax.axis("off")

        fig.colorbar(
            im,
            ax=axes,
            fraction=0.025,
            pad=0.02,
            label="Signed distance",
        )

        plt.tight_layout()
        plt.show()

    z_slider = widgets.IntSlider(
        value=n_slices // 2,
        min=0,
        max=n_slices - 1,
        step=1,
        description="Slice",
        continuous_update=False,
    )

    t_slider = widgets.FloatSlider(
        value=t_init,
        min=0.0,
        max=1.0,
        step=0.05,
        description="Tween",
        continuous_update=False,
    )

    clip_slider = widgets.FloatSlider(
        value=99,
        min=90,
        max=100,
        step=0.5,
        description="Clip %",
        continuous_update=False,
    )

    contour_toggle = widgets.Checkbox(
        value=True,
        description="Zero contour",
    )

    zoom_toggle = widgets.Checkbox(
        value=True,
        description="Zoom",
    )

    ui = widgets.VBox(
        [
            widgets.HBox([z_slider, t_slider]),
            widgets.HBox(
                [
                    clip_slider,
                    contour_toggle,
                    zoom_toggle,
                ]
            ),
        ]
    )

    out = widgets.interactive_output(
        update,
        {
            "z": z_slider,
            "t": t_slider,
            "clip_percentile": clip_slider,
            "show_zero_contour": contour_toggle,
            "zoom": zoom_toggle,
        },
    )

    display(ui, out)

In [26]:
interactive_sdf_tween_plotter(
    seg_a=data.gt,
    seg_b=data.observer_recontours[0],
    spacing=data.img_spacing,
    name_a="gt",
    name_b="Observer a",
    zoom_fraction = 0.15
)

Output()